In [82]:
import string
from tensorflow.keras.utils import to_categorical

PADDING_CHAR = '_'

ALPHANUM_CHARS = string.digits + string.ascii_uppercase + string.ascii_lowercase + PADDING_CHAR
CHAR_TO_INDEX = {char: i for i, char in enumerate(ALPHANUM_CHARS)}
INDEX_TO_CHAR = {i: char for char, i in CHAR_TO_INDEX.items()}
NUM_CLASSES = len(ALPHANUM_CHARS)



In [83]:
def load_alphanum_data(folder):
    X, y = [], []

    for filename in os.listdir(folder):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            filepath = os.path.join(folder, filename)
            img = cv2.imread(filepath)
            if img is None:
                continue
            img = cv2.resize(img, (64, 64))
            img = img.astype("float32") / 255.0
            X.append(img)

            label = os.path.splitext(filename)[0]  # Get label from filename
            label = label.ljust(MAX_LEN, PADDING_CHAR)[:MAX_LEN]  # pad/crop to MAX_LEN

            # 🔥 Convert each char to one-hot
            one_hot_label = [to_categorical(CHAR_TO_INDEX[c], num_classes=NUM_CLASSES) for c in label]
            y.append(one_hot_label)

    X = np.array(X)
    y = np.array(y)
    return X, y


In [84]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

def create_alphanum_model():
    input_img = Input(shape=(64, 64, 3))
    x = Conv2D(32, (3,3), activation='relu')(input_img)
    x = MaxPooling2D((2,2))(x)
    x = Conv2D(64, (3,3), activation='relu')(x)
    x = MaxPooling2D((2,2))(x)
    x = Flatten()(x)

    outputs = []
    for i in range(MAX_LEN):
        outputs.append(Dense(NUM_CLASSES, activation='softmax', name=f'char_{i+1}')(x))

    model = Model(inputs=input_img, outputs=outputs)
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [85]:
from tensorflow.keras.utils import to_categorical

y = np.array(y)
y_list = [to_categorical(y[:, i], num_classes=len(ALPHANUM_CHARS)) for i in range(MAX_LEN)]


In [90]:
X, y = load_alphanum_data(r"D:\Freelancing\bidding-bot\Captcha\subset")

# y shape: (num_samples, MAX_LEN, NUM_CLASSES)
# Now split this into a list of outputs (1 per char position)
y_list = [y[:, i] for i in range(MAX_LEN)]

model = create_alphanum_model()

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy'] * MAX_LEN
)

model.fit(X, y_list, batch_size=128, epochs=10, validation_split=0.1)

model.save("captcha_solver_alphanum.h5")



Epoch 1/10
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 514ms/step - char_1_accuracy: 0.6803 - char_1_loss: 2.0196 - char_2_accuracy: 0.0333 - char_2_loss: 3.8378 - char_3_accuracy: 0.0184 - char_3_loss: 4.4681 - char_4_accuracy: 0.0253 - char_4_loss: 4.4614 - char_5_accuracy: 0.0182 - char_5_loss: 4.4545 - char_6_accuracy: 0.6803 - char_6_loss: 1.8475 - char_7_accuracy: 0.6803 - char_7_loss: 1.8894 - char_8_accuracy: 0.6828 - char_8_loss: 1.7446 - char_9_accuracy: 0.6803 - char_9_loss: 1.8579 - loss: 26.6944 - val_char_1_accuracy: 1.0000 - val_char_1_loss: 3.6435e-04 - val_char_2_accuracy: 0.0000e+00 - val_char_2_loss: 12.6559 - val_char_3_accuracy: 0.0100 - val_char_3_loss: 4.6142 - val_char_4_accuracy: 0.0300 - val_char_4_loss: 4.8641 - val_char_5_accuracy: 0.0300 - val_char_5_loss: 4.5558 - val_char_6_accuracy: 1.0000 - val_char_6_loss: 3.2400e-04 - val_char_7_accuracy: 1.0000 - val_char_7_loss: 3.4915e-04 - val_char_8_accuracy: 1.0000 - val_char_8_loss: 3.5253e-04 - val_char_9_accuracy: 1.0000 - 

In [2]:
def decode_prediction(predictions):
    chars = []
    for pred in predictions:
        idx = np.argmax(pred)
        chars.append(IDX2CHAR[idx])
    return ''.join(chars).strip()  # Strip if padded with spaces


In [3]:
from tensorflow.keras.models import load_model

model = load_model("captcha_solver_alphanum.h5")
print("🔥 Model loaded again. Ready to wreck some CAPTCHAs.")


🔥 Model loaded again. Ready to wreck some CAPTCHAs.


In [4]:
from PIL import Image
import numpy as np

def preprocess_image(filepath):
    img = Image.open(filepath).convert('RGB')
    img = img.resize((64, 64))
    img_array = np.array(img) / 255.0  # Normalize
    return np.expand_dims(img_array, axis=0)  # Shape: (1, 64, 64, 3)


In [5]:
# Load and preprocess image
X_test = preprocess_image(r"D:\Freelancing\bidding-bot\Captcha\captcha\1AKDC.jpg")

# Make prediction
predictions = model.predict(X_test)

# Each prediction[i] is (1, NUM_CLASSES), so we get argmax per output
from tensorflow.keras.utils import to_categorical

predicted_label = ""
for i in range(len(predictions)):
    class_index = np.argmax(predictions[i])
    predicted_label += INDEX_TO_CHAR[class_index]

print("🔮 Predicted CAPTCHA:", predicted_label)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 585ms/step


NameError: name 'INDEX_TO_CHAR' is not defined